# MICrONS real-data lab: reciprocity among proofread neurons, pinned to v1507

**NeuroTrailblazers teaching notebook.** This is the real-data version of two offline
exercises: the pinned-snapshot query in *Tools and Methods* (Session 3) and the
reciprocity-versus-null analysis in *Algorithms and Applications* (Session 4). The
reasoning is the same. The difference is that the data are real, the version is a
real materialization, and some inclusion decisions now have consequences you can measure.

**Question.** Among MICrONS neurons whose axon *and* dendrite were both proofread,
are reciprocal connections (A→B and B→A) more common than expected under a null that
holds fixed the cell classes and the soma-to-soma distances?

**Data.** MICrONS cubic millimeter, datastack `minnie65_public`, **materialization
version 1507** (timestamp 2025-07-31 08:10:01 UTC). The tables are read from the
public static exports at `https://storage.googleapis.com/mat_dbs/public/minnie65_phase3_v1/v1507/`.
No account, token or cloud credentials are needed. The notebook checks each download
against a recorded SHA-256 hash and stops if the file differs.

**Citation (required by the data providers).** The MICrONS Consortium et al. 2025,
"Functional connectomics spanning multiple areas of mouse visual cortex," *Nature*
640: 435–447. Data: https://www.microns-explorer.org. The proofreading and cell-type
tables have their own papers; see the MICrONS tutorial's annotation-table reference.

**Runtime.** About 86 MB of downloads on the first run (cached afterwards). Analysis
takes a few minutes on a laptop, and needs about 2 GB of free memory.

## 1. Parameters (pinned; change nothing here for the reference run)

Every value that shapes the result lives in this cell. The endpoint, thresholds,
nulls and decision rule were fixed before the null comparisons were run. They were
not fixed before the data were opened: the lab author inspected table structure and
raw counts first. For your own run, treat them as pre-specified. Do not treat them as
a preregistered scientific hypothesis.

In [1]:
import os, json, hashlib, platform, datetime, gzip, io, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

MAT_VERSION = 1507          # primary materialization version. Never "latest".
DRIFT_VERSION = 1412        # earlier version, used only for the version-drift check
DATASTACK = "minnie65_public"
AGGREGATE = "minnie65_phase3_v1"
BASE_URL = "https://storage.googleapis.com/mat_dbs/public/minnie65_phase3_v1"
VOXEL_NM = np.array([4.0, 4.0, 40.0])   # pt_position units: 4 x 4 x 40 nm per voxel

PRIMARY_THRESHOLD = 1       # minimum synapses per directed pair to call an edge
SENSITIVITY_THRESHOLD = 2   # declared sensitivity analysis, not a second chance
DIST_BIN_UM = 25.0          # soma-distance bin width for the distance-and-type null
DIST_MAX_UM = 1000.0        # pairs beyond this share one overflow bin
N_NULL = 200                # Monte Carlo samples per null
SEED = 20250731             # fixed seed: the v1507 materialization date
ALPHA = 0.05                # upper-tail decision threshold

CACHE_DIR = Path(os.environ.get("MICRONS_LAB_CACHE",
                                Path.home() / ".cache" / "neurotrailblazers-microns-lab"))
OUTPUT_DIR = Path(os.environ.get("MICRONS_LAB_OUTPUT", "outputs"))

if not isinstance(MAT_VERSION, int) or not isinstance(DRIFT_VERSION, int):
    raise ValueError("Materialization versions must be explicit integers, never 'latest'.")

# Files this analysis reads, with the SHA-256 recorded on 2026-09-26.
FILES = {
    "synapses": (MAT_VERSION, "synapses_with_axon_proofreading.csv.gz",
                 "3f0841cafb39531e9b42d8a936211da66db9f09cea96c74ec88e0d05e3d06fd5"),
    "synapses_header": (MAT_VERSION, "synapses_with_axon_proofreading_header.csv", None),
    "proofreading": (MAT_VERSION, "proofreading_status_and_strategy_merged.csv.gz",
                     "d10300976fc3dd5fdb411dd0308dff75ee1385bc08b69639c9f8d7e970a1c4b1"),
    "proofreading_header": (MAT_VERSION, "proofreading_status_and_strategy_merged_header.csv", None),
    "cell_info": (MAT_VERSION, "aibs_cell_info_merged.csv.gz",
                  "f26299762ef15a782a9323e9b8ae0740b85bb50fa1be7b42fbecb4adfac1b7f8"),
    "cell_info_header": (MAT_VERSION, "aibs_cell_info_merged_header.csv", None),
    "proofreading_drift": (DRIFT_VERSION, "proofreading_status_and_strategy_merged.csv.gz",
                           "c580c50da233cb8e5c1090e97a21d145d608b188843348a2dc9e29ff387221c5"),
    "proofreading_drift_header": (DRIFT_VERSION, "proofreading_status_and_strategy_merged_header.csv", None),
}
RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
print("materialization", MAT_VERSION, "| drift check against", DRIFT_VERSION, "| run started", RUN_STARTED)

materialization 1507 | drift check against 1412 | run started 2026-09-26T14:59:09+00:00


## 2. Download and verify

Files are cached, so a rerun does not download them again. Each data file is hashed. If
a hash differs, the notebook stops rather than analysing different data under the same
version label. Header files are tiny and are hashed into the methods record, not checked.

In [2]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch(version, name, expected):
    url = f"{BASE_URL}/v{version}/{name}"
    dest = CACHE_DIR / f"v{version}" / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        tmp = dest.with_suffix(dest.suffix + ".part")
        with urllib.request.urlopen(url, timeout=120) as r, open(tmp, "wb") as f:
            while True:
                chunk = r.read(1 << 20)
                if not chunk:
                    break
                f.write(chunk)
        tmp.rename(dest)
    digest = sha256(dest)
    if expected is not None and digest != expected:
        raise RuntimeError(f"SHA-256 mismatch for {url}: got {digest}, expected {expected}. "
                           "The file changed; do not report results under this version label.")
    return {"url": url, "path": str(dest), "bytes": dest.stat().st_size, "sha256": digest,
            "hash_checked": expected is not None}


provenance = {key: fetch(v, name, exp) for key, (v, name, exp) in FILES.items()}
for key, p in provenance.items():
    print(f"{key:26s} {p['bytes']:>12,d} bytes  sha256 {p['sha256'][:16]}…  checked={p['hash_checked']}")

synapses                     80,191,720 bytes  sha256 3f0841cafb39531e…  checked=True
synapses_header                     271 bytes  sha256 e9265a2ba4537ebe…  checked=False
proofreading                     69,077 bytes  sha256 d10300976fc3dd5f…  checked=True
proofreading_header                 223 bytes  sha256 0576b8b864f3441b…  checked=False
cell_info                     5,696,726 bytes  sha256 f26299762ef15a78…  checked=True
cell_info_header                    372 bytes  sha256 48551ea524e8567a…  checked=False
proofreading_drift               63,838 bytes  sha256 c580c50da233cb8e…  checked=True
proofreading_drift_header           223 bytes  sha256 0576b8b864f3441b…  checked=False


In [3]:
def read_table(key):
    """Read a headerless CSV export using its companion header file (name,type per line)."""
    header = Path(provenance[key + "_header"]["path"]).read_text().strip().splitlines()
    names = [line.split(",")[0] for line in header]
    return pd.read_csv(provenance[key]["path"], names=names, header=None)


syn = read_table("synapses")[["pre_pt_root_id", "post_pt_root_id", "size", "strategy_axon"]]
proof = read_table("proofreading")
cells = read_table("cell_info")
proof_drift = read_table("proofreading_drift")
print(f"synapses from proofread axons: {len(syn):,} rows, {syn.pre_pt_root_id.nunique():,} presynaptic cells")
print(f"proofreading table v{MAT_VERSION}: {len(proof):,} rows;  v{DRIFT_VERSION}: {len(proof_drift):,} rows")
print(f"cell info v{MAT_VERSION}: {len(cells):,} nucleus rows")

synapses from proofread axons: 2,089,627 rows, 2,141 presynaptic cells
proofreading table v1507: 2,182 rows;  v1412: 2,020 rows
cell info v1507: 144,120 nucleus rows


## 3. Inclusion rules

These rules are applied in order, and every exclusion is counted. The rules are:

1. **Axon proofread** (`status_axon == 't'`). Without this, missing outputs look like
   missing connections.
2. **Dendrite proofread** (`status_dendrite == 't'`). Without this, missing inputs, or
   inputs added by a merge, would distort the counts.
3. **Proofread state is current** (`valid_id == pt_root_id`). The cell must not have
   been edited since its proofreading was assessed. The MICrONS documentation defines
   `valid_id` as the root ID at the time of that assessment.
4. **Exactly one nucleus on the root ID.** A root ID with zero or several nuclei is a
   fragment or a merge. Its cell-class label is ambiguous.
5. **Cell class is excitatory or inhibitory** (`broad_type` in the v1507 cell-info
   table). The label's origin is recorded in `broad_type_source`. Some labels are
   manual reference calls for the V1 column. Others are classifier predictions. The
   notebook prints the mix; neither kind is ground truth.

Rules 4 and 5 may exclude nothing at v1507, because rules 1–3 already remove the
problem cases. Keep them anyway: they are guards against a different version or a
different cell table.

A directed edge A→B exists if at least *T* synapses run from A's axon to B, with both
A and B in the included set. Autapses (A = B) are dropped. The primary threshold is
T = 1. The declared sensitivity threshold is T = 2.

In [4]:
steps = []
P = proof.copy()
steps.append(("all rows in proofreading table", len(P)))
P = P[P.status_axon == "t"];                steps.append(("1. axon proofread", len(P)))
P = P[P.status_dendrite == "t"];            steps.append(("2. dendrite proofread", len(P)))
P = P[P.valid_id == P.pt_root_id];          steps.append(("3. valid_id == pt_root_id", len(P)))

nuclei_per_root = cells[cells.pt_root_id != 0].pt_root_id.value_counts()
P = P[P.pt_root_id.map(nuclei_per_root).fillna(0).astype(int) == 1]
steps.append(("4. exactly one nucleus on root", len(P)))

one_nucleus = cells[cells.pt_root_id.isin(P.pt_root_id)].set_index("pt_root_id")
P = P.assign(broad_type=P.pt_root_id.map(one_nucleus.broad_type),
             broad_type_source=P.pt_root_id.map(one_nucleus.broad_type_source),
             visual_area=P.pt_root_id.map(one_nucleus.visual_area))
P = P[P.broad_type.isin(["excitatory", "inhibitory"])]
steps.append(("5. broad_type excitatory or inhibitory", len(P)))
P = P.sort_values("pt_root_id").reset_index(drop=True)

inclusion = pd.DataFrame(steps, columns=["step", "cells_remaining"])
inclusion["excluded_at_step"] = (-inclusion.cells_remaining.diff()).fillna(0).astype(int)
print(inclusion.to_string(index=False))
print()
print(pd.crosstab(P.strategy_axon, P.broad_type, margins=True))
print()
print("visual area:", P.visual_area.value_counts().to_dict())
print("class label source:", P.broad_type_source.value_counts().to_dict())

                                  step  cells_remaining  excluded_at_step
        all rows in proofreading table             2182                 0
                     1. axon proofread             2141                41
                 2. dendrite proofread             2088                53
             3. valid_id == pt_root_id             2070                18
        4. exactly one nucleus on root             2070                 0
5. broad_type excitatory or inhibitory             2070                 0

broad_type               excitatory  inhibitory   All
strategy_axon                                        
axon_fully_extended             111         121   232
axon_interareal                 123           1   124
axon_partially_extended        1498         216  1714
All                            1732         338  2070

visual area: {'V1': 1854, 'RL': 156, 'AL': 59, 'LM': 1}
class label source: {'allen_v1_column_types_slanted_ref': 1341, 'baylor_log_reg_cell_type_coarse_v1'

## 4. Build the directed graph

Keep only synapses whose presynaptic *and* postsynaptic root IDs are both in the
included set, and count synapses per ordered pair. Then threshold those counts.

In [5]:
ids = P.pt_root_id.to_numpy()
index = pd.Series(np.arange(len(ids)), index=ids)
n = len(ids)

within = syn[syn.pre_pt_root_id.isin(index.index) & syn.post_pt_root_id.isin(index.index)]
autapses = int((within.pre_pt_root_id == within.post_pt_root_id).sum())
within = within[within.pre_pt_root_id != within.post_pt_root_id]
pair_counts = within.groupby(["pre_pt_root_id", "post_pt_root_id"]).size()

W = np.zeros((n, n), dtype=np.int32)
W[index[pair_counts.index.get_level_values(0)].to_numpy(),
  index[pair_counts.index.get_level_values(1)].to_numpy()] = pair_counts.to_numpy()

is_exc = (P.broad_type == "excitatory").to_numpy()
print(f"included cells n = {n:,}  ({is_exc.sum():,} E, {(~is_exc).sum():,} I)")
print(f"synapses among included cells: {len(within):,}  (autapses dropped: {autapses:,})")
print(f"ordered pairs with >=1 synapse: {(W >= 1).sum():,};  with >=2: {(W >= 2).sum():,}")
print("synapses per connected pair:", pd.Series(W[W > 0]).value_counts().sort_index().head(8).to_dict())

included cells n = 2,070  (1,732 E, 338 I)
synapses among included cells: 258,812  (autapses dropped: 31,542)
ordered pairs with >=1 synapse: 134,753;  with >=2: 45,774
synapses per connected pair: {1: 88979, 2: 20850, 3: 9211, 4: 5298, 5: 3256, 6: 2092, 7: 1453, 8: 933}


## 5. Descriptive endpoint: connection probability by class pair

This is the observed fraction of ordered pairs that are connected. It is a
description, not a test. Most of these pairs are far apart, so the fractions are
diluted by distance.

In [6]:
def class_pair_table(A):
    rows = []
    for pre_name, pre_mask in (("E", is_exc), ("I", ~is_exc)):
        for post_name, post_mask in (("E", is_exc), ("I", ~is_exc)):
            sub = A[np.ix_(pre_mask, post_mask)]
            possible = pre_mask.sum() * post_mask.sum() - (pre_mask & post_mask).sum()
            rows.append({"pre": pre_name, "post": post_name, "connected_pairs": int(sub.sum()),
                         "possible_pairs": int(possible), "fraction_connected": sub.sum() / possible})
    return pd.DataFrame(rows)


class_tables = []
for T in (PRIMARY_THRESHOLD, SENSITIVITY_THRESHOLD):
    t = class_pair_table(W >= T).assign(threshold=T)
    class_tables.append(t)
class_pairs = pd.concat(class_tables, ignore_index=True)
print(class_pairs.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

pre post  connected_pairs  possible_pairs  fraction_connected  threshold
  E    E            41094         2998092              0.0137          1
  E    I            31067          585416              0.0531          1
  I    E            51361          585416              0.0877          1
  I    I            11231          113906              0.0986          1
  E    E             4005         2998092              0.0013          2
  E    I             9717          585416              0.0166          2
  I    E            26392          585416              0.0451          2
  I    I             5660          113906              0.0497          2


## 6. Primary endpoint: reciprocal pairs against three nulls

The same two quantities as the Session 4 worksheet:

- **reciprocal pairs**, meaning unordered pairs with both directed edges, each counted once;
- **reciprocated-edge fraction**, meaning the fraction of directed edges whose reverse edge exists.

Each null holds a different structure fixed:

| Null | Holds fixed | Leaves free |
|---|---|---|
| Uniform fixed-edge-count | nodes, number of edges | degrees, distance, class |
| Soft configuration (Chung–Lu) | each cell's in- and out-degree, in expectation | distance, class |
| Distance × class | connection probability per class pair per 25 µm distance bin | degrees |

**Decision rule (fixed before running the nulls).** Call reciprocity *enriched* only
if the upper-tail Monte Carlo probability is at most 0.05 under the **distance × class
null** at T = 1, and the observed-to-expected ratio is above 1 at T = 2. The other two
nulls are shown so you can see how much of the apparent effect each constraint absorbs.

In [7]:
rng = np.random.default_rng(SEED)

pos_um = P[["pt_position_x", "pt_position_y", "pt_position_z"]].to_numpy(float) * VOXEL_NM / 1000.0
diff = pos_um[:, None, :] - pos_um[None, :, :]
dist_um = np.sqrt((diff ** 2).sum(-1))
del diff
dist_bin = np.minimum((dist_um // DIST_BIN_UM).astype(np.int32), int(DIST_MAX_UM // DIST_BIN_UM))
class_code = np.where(is_exc, 0, 1)
STRATUM = (class_code[:, None] * 2 + class_code[None, :]) * 1000 + dist_bin   # class pair x distance bin


def reciprocal_pairs(A):
    return int(np.triu(A & A.T, k=1).sum())


def upper_tail(null_counts, observed):
    return (1 + int((null_counts >= observed).sum())) / (1 + len(null_counts))


def probs_chung_lu(A, stratum):
    kout, kin = A.sum(1).astype(float), A.sum(0).astype(float)
    Pm = np.minimum(1.0, np.outer(kout, kin) / A.sum())
    np.fill_diagonal(Pm, 0.0)
    return Pm


def probs_distance_class(A, stratum):
    offdiag = ~np.eye(A.shape[0], dtype=bool)
    s = stratum[offdiag]
    p_stratum = pd.Series(A[offdiag].astype(np.int64)).groupby(s).mean()
    Pm = np.zeros(A.shape)
    Pm[offdiag] = p_stratum.reindex(s).to_numpy()
    return Pm


def sample_bernoulli(Pm, k):
    return np.array([reciprocal_pairs(rng.random(Pm.shape) < Pm) for _ in range(k)])


def sample_uniform(A, k):
    size, m = A.shape[0], int(A.sum())
    free = np.flatnonzero(~np.eye(size, dtype=bool).ravel())
    out = np.empty(k, dtype=np.int64)
    for i in range(k):
        B = np.zeros(size * size, dtype=bool)
        B[rng.choice(free, size=m, replace=False)] = True
        out[i] = reciprocal_pairs(B.reshape(size, size))
    return out


def analyse(A, stratum, label):
    size, m = A.shape[0], int(A.sum())
    obs = reciprocal_pairs(A)
    N2 = size * (size - 1)
    nulls = [("uniform fixed-edge-count",
              (N2 / 2) * m * (m - 1) / (N2 * (N2 - 1)),     # exact expectation
              sample_uniform(A, N_NULL))]
    for name, fn in (("soft configuration (Chung-Lu)", probs_chung_lu),
                     ("distance x class", probs_distance_class)):
        Pm = fn(A, stratum)
        nulls.append((name, float(np.triu(Pm * Pm.T, k=1).sum()), sample_bernoulli(Pm, N_NULL)))
    return [{"analysis": label, "null": name, "n_cells": size, "edges": m,
             "observed_reciprocal_pairs": obs, "reciprocated_edge_fraction": 2 * obs / m,
             "expected_reciprocal_pairs": exp, "observed_over_expected": obs / exp,
             "null_mean": float(null.mean()), "null_sd": float(null.std(ddof=1)),
             "null_max": int(null.max()), "upper_tail_p": upper_tail(null, obs),
             "n_null_samples": len(null)} for name, exp, null in nulls]


results = []
for T in (PRIMARY_THRESHOLD, SENSITIVITY_THRESHOLD):
    results += analyse(W >= T, STRATUM, f"threshold {T}")
recip = pd.DataFrame(results)
print(recip.drop(columns=["n_cells", "n_null_samples"]).to_string(
    index=False, float_format=lambda x: f"{x:.4g}"))

   analysis                          null  edges  observed_reciprocal_pairs  reciprocated_edge_fraction  expected_reciprocal_pairs  observed_over_expected  null_mean  null_sd  null_max  upper_tail_p
threshold 1      uniform fixed-edge-count 134753                      17022                      0.2526                       2120                    8.03       2116    41.56      2218      0.004975
threshold 1 soft configuration (Chung-Lu) 134753                      17022                      0.2526                       8105                     2.1       8106    79.41      8315      0.004975
threshold 1              distance x class 134753                      17022                      0.2526                       8652                   1.967       8641    96.24      8871      0.004975
threshold 2      uniform fixed-edge-count  45774                       5542                      0.2421                      244.6                   22.66      244.2    15.16       283      0.004975
thres

## 7. Second sensitivity analysis: the strictest axon strategy only

The inclusion set mixes axon proofreading strategies. A **partially extended** axon
was followed outward, but not every ending was extended, so some of its true outputs
are missing. Restrict the set to cells with `strategy_axon == 'axon_fully_extended'`
and ask whether the direction of the result survives. The sample is smaller and
inhibitory cells are over-represented, so read this as a check on direction, not as
a second estimate. The distance × class null is re-estimated on this subset.

In [8]:
full = np.flatnonzero((P.strategy_axon == "axon_fully_extended").to_numpy())
sub = np.ix_(full, full)
strict = analyse((W >= PRIMARY_THRESHOLD)[sub], STRATUM[sub], "fully extended axons, threshold 1")
recip = pd.concat([recip, pd.DataFrame(strict)], ignore_index=True)
print(f"cells with fully extended axons: {len(full)} ({is_exc[full].sum()} E, {(~is_exc[full]).sum()} I)")
print(pd.DataFrame(strict).drop(columns=["n_null_samples"]).to_string(index=False, float_format=lambda x: f"{x:.4g}"))

cells with fully extended axons: 232 (111 E, 121 I)
                         analysis                          null  n_cells  edges  observed_reciprocal_pairs  reciprocated_edge_fraction  expected_reciprocal_pairs  observed_over_expected  null_mean  null_sd  null_max  upper_tail_p
fully extended axons, threshold 1      uniform fixed-edge-count      232   6540                       1153                      0.3526                        399                    2.89      398.5    17.96       444      0.004975
fully extended axons, threshold 1 soft configuration (Chung-Lu)      232   6540                       1153                      0.3526                      627.9                   1.836      629.5    21.88       683      0.004975
fully extended axons, threshold 1              distance x class      232   6540                       1153                      0.3526                      771.3                   1.495      772.7    24.99       854      0.004975


## 8. Apply the decision rule

In [9]:
def row(analysis, null):
    return recip[(recip.analysis == analysis) & (recip.null == null)].iloc[0]

primary = row(f"threshold {PRIMARY_THRESHOLD}", "distance x class")
sens = row(f"threshold {SENSITIVITY_THRESHOLD}", "distance x class")
strict_row = row("fully extended axons, threshold 1", "distance x class")
enriched = bool(primary.upper_tail_p <= ALPHA and sens.observed_over_expected > 1)
decision = ("enriched under the distance x class null" if enriched
            else "not established under the distance x class null")
print(f"primary (T={PRIMARY_THRESHOLD}): observed {primary.observed_reciprocal_pairs}, "
      f"expected {primary.expected_reciprocal_pairs:.1f}, ratio {primary.observed_over_expected:.2f}, "
      f"p <= {primary.upper_tail_p:.4f}")
print(f"sensitivity (T={SENSITIVITY_THRESHOLD}): ratio {sens.observed_over_expected:.2f}, p <= {sens.upper_tail_p:.4f}")
print(f"strict inclusion: ratio {strict_row.observed_over_expected:.2f}, p <= {strict_row.upper_tail_p:.4f}")
print("DECISION:", decision)
print("Note: with", N_NULL, "samples the smallest reportable p is", round(1 / (N_NULL + 1), 4))

primary (T=1): observed 17022, expected 8652.3, ratio 1.97, p <= 0.0050
sensitivity (T=2): ratio 3.04, p <= 0.0050
strict inclusion: ratio 1.49, p <= 0.0050
DECISION: enriched under the distance x class null
Note: with 200 samples the smallest reportable p is 0.005


## 9. Version drift: the same inclusion table at v1412 and v1507

The Session 3 worksheet used two invented snapshots. Here the two snapshots are real.
Rows are matched on `pt_supervoxel_id`, the supervoxel under the nucleus point, because
supervoxel IDs do not change when the segmentation is edited. Root IDs do change. The
check asks how many cells entered or left the proofread set between versions, and how
many kept their identity but changed root ID. A v1412 root ID that changed would
silently match nothing in the v1507 synapse table.

The synapse export used above exists only at v1507 in this static form, so this lab
does not rebuild the whole graph at v1412. That is an extension for learners with a CAVE token.

In [10]:
def proofread_both(df):
    return df[(df.status_axon == "t") & (df.status_dendrite == "t") & (df.valid_id == df.pt_root_id)]

a, b = proofread_both(proof_drift), proofread_both(proof)
a_sv, b_sv = a.set_index("pt_supervoxel_id"), b.set_index("pt_supervoxel_id")
shared = a_sv.index.intersection(b_sv.index)
root_changed = int((a_sv.loc[shared, "pt_root_id"] != b_sv.loc[shared, "pt_root_id"]).sum())
stale = int((~a.pt_root_id.isin(proof.pt_root_id)).sum())
drift = {
    "earlier_version": DRIFT_VERSION, "later_version": MAT_VERSION,
    "rows_earlier": int(len(proof_drift)), "rows_later": int(len(proof)),
    "proofread_both_current_earlier": int(len(a)), "proofread_both_current_later": int(len(b)),
    "matched_on_supervoxel": int(len(shared)),
    "only_in_earlier": int(len(a_sv.index.difference(b_sv.index))),
    "only_in_later": int(len(b_sv.index.difference(a_sv.index))),
    "matched_but_root_id_changed": root_changed,
    "earlier_root_ids_absent_from_later_table": stale,
}
for k, v in drift.items():
    print(f"{k:45s} {v}")

earlier_version                               1412
later_version                                 1507
rows_earlier                                  2020
rows_later                                    2182
proofread_both_current_earlier                1953
proofread_both_current_later                  2070
matched_on_supervoxel                         1942
only_in_earlier                               11
only_in_later                                 128
matched_but_root_id_changed                   118
earlier_root_ids_absent_from_later_table      129


## 10. Methods record and archived outputs

The record identifies the code by hashing the source of this notebook's code cells.
Outputs are excluded from the hash, so an executed copy gets the same code hash as a
clean one. Compare your `results_summary.json` with the archived copy on the lab page.

In [11]:
nb_path = Path(os.environ.get("MICRONS_LAB_NOTEBOOK", "microns-lab.ipynb"))
if nb_path.exists():
    nb = json.loads(nb_path.read_text())
    code_src = "\n".join("".join(c["source"]) if isinstance(c["source"], list) else c["source"]
                         for c in nb["cells"] if c["cell_type"] == "code")
    code_hash = hashlib.sha256(code_src.encode()).hexdigest()
else:
    code_hash = "unknown (notebook file not found from working directory)"

import importlib.metadata as md_
versions = {pkg: md_.version(pkg) for pkg in ("numpy", "pandas")}
versions["python"] = platform.python_version()

methods = {
    "dataset": "MICrONS cubic millimeter (minnie65)",
    "datastack": DATASTACK,
    "materialization_version": MAT_VERSION,
    "materialization_timestamp_utc": "2025-07-31T08:10:01.117494+00:00",
    "access": "public static CSV exports, no account or token",
    "files": {k: {kk: v[kk] for kk in ("url", "bytes", "sha256", "hash_checked")} for k, v in provenance.items()},
    "tables": ["synapses_with_axon_proofreading", "proofreading_status_and_strategy", "aibs_cell_info"],
    "inclusion_rules": [s for s, _ in steps[1:]],
    "edge_rule": "directed edge if >= T synapses from pre axon to post, both included, autapses dropped",
    "thresholds": {"primary": PRIMARY_THRESHOLD, "sensitivity": SENSITIVITY_THRESHOLD},
    "nulls": ["uniform fixed-edge-count", "soft configuration (Chung-Lu)",
              f"distance x class ({DIST_BIN_UM:g} um bins to {DIST_MAX_UM:g} um, then one overflow bin)"],
    "decision_rule": "enriched iff upper-tail p <= 0.05 under distance x class null at T=1 and ratio > 1 at T=2",
    "n_null_samples": N_NULL, "seed": SEED,
    "voxel_resolution_nm": VOXEL_NM.tolist(),
    "code_sha256_code_cells": code_hash,
    "package_versions": versions,
    "platform": platform.platform(),
    "run_started_utc": RUN_STARTED,
    "citation": "MICrONS Consortium et al. 2025, Nature 640:435-447; www.microns-explorer.org",
}

summary = {
    "materialization_version": MAT_VERSION,
    "inclusion": inclusion.to_dict(orient="records"),
    "n_cells": int(n), "n_excitatory": int(is_exc.sum()), "n_inhibitory": int((~is_exc).sum()),
    "synapses_among_included": int(len(within)), "autapses_dropped": autapses,
    "edges": {f"threshold_{T}": int((W >= T).sum()) for T in (PRIMARY_THRESHOLD, SENSITIVITY_THRESHOLD)},
    "reciprocity": recip.round(6).to_dict(orient="records"),
    "decision": decision,
    "version_drift": drift,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "methods_record.json").write_text(json.dumps(methods, indent=2) + "\n")
(OUTPUT_DIR / "results_summary.json").write_text(json.dumps(summary, indent=2) + "\n")
inclusion.to_csv(OUTPUT_DIR / "inclusion_exclusions.csv", index=False)
class_pairs.to_csv(OUTPUT_DIR / "class_pair_connectivity.csv", index=False, float_format="%.6g")
recip.to_csv(OUTPUT_DIR / "reciprocity_nulls.csv", index=False, float_format="%.6g")
print("wrote", sorted(p.name for p in OUTPUT_DIR.iterdir()))
print(json.dumps({k: methods[k] for k in ("materialization_version", "code_sha256_code_cells", "package_versions")}, indent=2))

wrote ['class_pair_connectivity.csv', 'inclusion_exclusions.csv', 'methods_record.json', 'reciprocity_nulls.csv', 'results_summary.json']
{
  "materialization_version": 1507,
  "code_sha256_code_cells": "a50d89761feb6371a28f2b8d977836bdc95066afd96469acf2d122de3f68f0c3",
  "package_versions": {
    "numpy": "2.3.5",
    "pandas": "2.3.3",
    "python": "3.13.5"
  }
}


## 11. Limitations. Read these before writing any sentence about biology.

- **Selection.** Proofread cells were not sampled at random. Most sit in or near the
  V1 column that was proofread first. Their axons were extended with different
  strategies and for different projects. The result describes this set.
- **Incomplete axons.** Most included axons are *partially extended*. Missing axon
  branches remove true edges, and reciprocal pairs lose two chances to be seen, not one.
  Section 7 checks direction on the smaller fully extended set. It cannot recover
  what is missing.
- **Synapse detection.** Synapses are automatically detected. Single-synapse edges are
  the most exposed to false positives, so threshold 2 is reported alongside.
- **Class labels.** E/I labels come from several source tables. Some are manual
  column reference calls and some are classifier predictions. A misclassified cell
  moves its pairs into the wrong stratum of the distance × class null.
- **Autapses and self-contacts.** Synapses from a cell onto itself are dropped. There
  are many of them, and most are probably detection or segmentation artifacts, but
  this lab does not check.
- **Volume boundary.** Cells near the edge of the volume lose arbor, and so lose
  connections, outside it. No boundary correction is applied.
- **Null scope.** The distance × class null uses soma-to-soma distance. It ignores
  axon–dendrite overlap, layer and cell subtype, and it does not preserve each cell's
  degree. The Chung–Lu null preserves degree but ignores distance. Neither null holds
  both fixed at once. A ratio above 1 under this null says
  only that these constraints do not account for the reciprocity. It does not identify
  a mechanism.
- **Version.** Everything here is conditional on v1507. Proofreading continues after
  it, as Section 9 shows.